# Pseudobulk model building — MOFA-FLEX with BP prior

**Environment:** `clamp-analyses`

Trains MOFA-FLEX (Horseshoe prior + GO Biological Process pathway annotations) on every pseudobulk dataset. Reads the z-scored expression matrix (`norm.csv`) and rank estimate (`k.csv`) written by `01_CLAMP.ipynb`. Outputs written to `output/01_model_building/05_pseudobulk/<dataset>/MOFA_FLEX_PRIOR/`.

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import mofaflex as mfl
import pickle
from pathlib import Path
from pyprojroot.here import here

## Configuration

In [ ]:
DATASET    = "PBMC_Perez2022"
OUT_ROOT   = "output/01_model_building/05_pseudobulk"
MAX_EPOCHS = 200
SEED       = 123

## Build MOFA-FLEX model for each dataset

In [ ]:
print(f"========== {DATASET} ==========")
from pyprojroot.here import here as _here
from pathlib import Path
ds_dir = Path(_here(OUT_ROOT)) / DATASET

# Load preprocessed data and k
norm_path = ds_dir / "norm.csv"
if not norm_path.exists():
    raise FileNotFoundError(f"{norm_path} not found — run 00_preprocess.ipynb first.")

norm = pd.read_csv(norm_path, index_col=0).astype(np.float32)
k    = int(pd.read_csv(ds_dir / "k.csv")["k"].iloc[0])
print(f"  norm shape: {norm.shape}  k={k}")

gene_list    = norm.index.tolist()
sample_names = norm.columns.tolist()

# Build BP pathway annotations from MSigDB
print("  Building BP pathway annotations ...")
bp_collection = mfl.tl.msigdb_get_features(category="c5.go.bp", dbver="2026.1.Hs")
bp_collection = bp_collection.filter(
    gene_list, min_fraction=0.4, min_count=40, max_count=200
)
bp_collection = bp_collection.merge_similar(
    metric="jaccard", similarity_threshold=0.8, iteratively=True
)
print(f"  Filtered BP pathways: {len(bp_collection)}")

# Build AnnData (samples x genes)
adata = ad.AnnData(
    X   = norm.T.values,
    obs = pd.DataFrame(index=sample_names),
    var = pd.DataFrame(index=gene_list)
)
adata.varm["annotations"] = bp_collection.to_mask(gene_list).T
print(f"  AnnData: {adata.shape}  annotations: {adata.varm['annotations'].shape}")

# MOFA-FLEX model
print(f"  Training MOFA-FLEX (n_factors={k}, max_epochs={MAX_EPOCHS}) ...")
data_opts = mfl.DataOptions(
    scale_per_group        = False,
    plot_data_overview     = False,
    annotations_varm_key   = "annotations"
)
model_opts = mfl.ModelOptions(
    n_factors    = k,
    weight_prior = "Horseshoe",
    likelihoods  = "Normal"
)
train_opts = mfl.TrainingOptions(
    seed       = SEED,
    max_epochs = MAX_EPOCHS,
    save_path  = False,
    device     = "cpu"
)
model = mfl.MOFAFLEX(
    {"group_1": {"view_1": adata}},
    data_opts, model_opts, train_opts
)

# Extract and save results
factors = model.get_factors()["group_1"]
weights = model.get_weights()["view_1"]

B_matrix = factors.T
B_matrix.columns = sample_names
B_matrix.index   = [f"LV{i+1}" for i in range(len(B_matrix))]

out_dir = ds_dir / "MOFA_FLEX_PRIOR"
out_dir.mkdir(parents=True, exist_ok=True)
B_matrix.to_csv(out_dir / "B_matrix.csv")
weights.to_csv(out_dir / "Z_matrix.csv")
with open(out_dir / "model.pkl", "wb") as f:
    pickle.dump(model, f)

print(f"  MOFA-FLEX saved -> {out_dir}")